# 🔄 02 — ETL Pipeline & Feature Engineering

**E-Commerce Customer Behavior Analysis & Hybrid Recommendation System**

---

## Pipeline Overview
1. **Extract** — Load raw datasets
2. **Transform** — Clean, handle missing values, outliers, feature engineering
3. **Load** — Populate MySQL Star Schema + generate merged dataset

In [ ]:
# ============================================================
# Imports
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Paths
DATA_RAW = Path('../data/raw')
DATA_PROCESSED = Path('../data/processed')
DATA_GENERATED = Path('../data/generated')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_GENERATED.mkdir(parents=True, exist_ok=True)

print('✅ Ready')

---
## PART 1: EXTRACT — Load Raw Data

In [ ]:
# ============================================================
# 1.1 Load Online Retail II
# ============================================================
retail_path = DATA_RAW / 'online_retail' / 'online_retail_II.xlsx'
df_y1 = pd.read_excel(retail_path, sheet_name='Year 2009-2010')
df_y2 = pd.read_excel(retail_path, sheet_name='Year 2010-2011')
df_retail = pd.concat([df_y1, df_y2], ignore_index=True)
print(f'Retail: {df_retail.shape}')
df_retail.head(2)

In [ ]:
# ============================================================
# 1.2 Load Reviews
# ============================================================
df_reviews = pd.read_csv(DATA_RAW / 'clothing_reviews' / 'Womens Clothing E-Commerce Reviews.csv')
print(f'Reviews: {df_reviews.shape}')
df_reviews.head(2)

In [ ]:
# ============================================================
# 1.3 Load Amazon Products (main file)
# ============================================================
df_amazon = pd.read_csv(DATA_RAW / 'amazon_products' / 'Amazon-Products.csv')
print(f'Amazon: {df_amazon.shape}')
print(f'Columns: {list(df_amazon.columns)}')
df_amazon.head(2)

---
## PART 2: TRANSFORM — Data Cleaning

### 2.1 Clean Online Retail II

In [ ]:
# ============================================================
# 2.1.1 Before cleaning stats
# ============================================================
print(f'Before cleaning: {df_retail.shape}')
print(f'Missing values:\n{df_retail.isnull().sum()}')
print(f'\nDuplicates: {df_retail.duplicated().sum()}')

In [ ]:
# ============================================================
# 2.1.2 Remove cancelled orders (Invoice starts with 'C')
# ============================================================
df_retail['Invoice'] = df_retail['Invoice'].astype(str)
cancelled = df_retail[df_retail['Invoice'].str.startswith('C')]
print(f'Cancelled orders: {len(cancelled)} ({len(cancelled)/len(df_retail)*100:.1f}%)')

# Save cancellations separately for analysis
cancelled.to_csv(DATA_PROCESSED / 'retail_cancellations.csv', index=False)

# Keep only valid orders
df_retail = df_retail[~df_retail['Invoice'].str.startswith('C')].copy()
print(f'After removing cancellations: {df_retail.shape}')

In [ ]:
# ============================================================
# 2.1.3 Drop rows with missing Customer ID
# ============================================================
null_customers = df_retail['Customer ID'].isnull().sum()
print(f'Rows with null Customer ID: {null_customers} ({null_customers/len(df_retail)*100:.1f}%)')
df_retail = df_retail.dropna(subset=['Customer ID']).copy()
df_retail['Customer ID'] = df_retail['Customer ID'].astype(int)
print(f'After dropping null customers: {df_retail.shape}')

In [ ]:
# ============================================================
# 2.1.4 Remove duplicates
# ============================================================
dupes = df_retail.duplicated().sum()
print(f'Duplicate rows: {dupes}')
df_retail = df_retail.drop_duplicates().copy()
print(f'After dedup: {df_retail.shape}')

In [ ]:
# ============================================================
# 2.1.5 Filter negative/zero Quantity and Price
# ============================================================
neg_qty = (df_retail['Quantity'] <= 0).sum()
neg_price = (df_retail['Price'] <= 0).sum()
print(f'Negative/zero Quantity: {neg_qty}')
print(f'Negative/zero Price: {neg_price}')

df_retail = df_retail[(df_retail['Quantity'] > 0) & (df_retail['Price'] > 0)].copy()
print(f'After filtering: {df_retail.shape}')

In [ ]:
# ============================================================
# 2.1.6 Handle outliers using IQR method
# ============================================================
def remove_outliers_iqr(df, column, factor=1.5):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - factor * IQR
    upper = Q3 + factor * IQR
    before = len(df)
    df = df[(df[column] >= lower) & (df[column] <= upper)].copy()
    removed = before - len(df)
    print(f'  {column}: removed {removed} outliers (range: {lower:.2f} to {upper:.2f})')
    return df

print('Removing outliers (IQR method):')
df_retail = remove_outliers_iqr(df_retail, 'Quantity')
df_retail = remove_outliers_iqr(df_retail, 'Price')
print(f'After outlier removal: {df_retail.shape}')

In [ ]:
# ============================================================
# 2.1.7 Compute TotalPrice & extract time features
# ============================================================
df_retail['TotalPrice'] = df_retail['Quantity'] * df_retail['Price']
df_retail['InvoiceDate'] = pd.to_datetime(df_retail['InvoiceDate'])
df_retail['Year'] = df_retail['InvoiceDate'].dt.year
df_retail['Month'] = df_retail['InvoiceDate'].dt.month
df_retail['DayOfWeek'] = df_retail['InvoiceDate'].dt.day_name()
df_retail['Hour'] = df_retail['InvoiceDate'].dt.hour
df_retail['IsWeekend'] = df_retail['InvoiceDate'].dt.dayofweek.isin([5, 6]).astype(int)
df_retail['Quarter'] = df_retail['InvoiceDate'].dt.quarter

print('Time features added ✅')
print(f'Date range: {df_retail["InvoiceDate"].min()} → {df_retail["InvoiceDate"].max()}')
df_retail.head(3)

In [ ]:
# ============================================================
# 2.1.8 Clean Description column
# ============================================================
df_retail['Description'] = df_retail['Description'].fillna('UNKNOWN')
df_retail['Description'] = df_retail['Description'].str.strip().str.upper()
print(f'Unique products: {df_retail["StockCode"].nunique()}')
print(f'Unique descriptions: {df_retail["Description"].nunique()}')

In [ ]:
# ============================================================
# 2.1.9 Summary after cleaning
# ============================================================
print('=== RETAIL CLEANING SUMMARY ===')
print(f'Final shape: {df_retail.shape}')
print(f'Missing values: {df_retail.isnull().sum().sum()}')
print(f'Customers: {df_retail["Customer ID"].nunique()}')
print(f'Products: {df_retail["StockCode"].nunique()}')
print(f'Countries: {df_retail["Country"].nunique()}')
print(f'Total Revenue: £{df_retail["TotalPrice"].sum():,.2f}')

# Save cleaned retail
df_retail.to_csv(DATA_PROCESSED / 'retail_clean.csv', index=False)
print('\n✅ Saved: data/processed/retail_clean.csv')

### 2.2 Clean Reviews Dataset

In [ ]:
# ============================================================
# 2.2.1 Inspect & clean reviews
# ============================================================
print(f'Before: {df_reviews.shape}')
print(f'Columns: {list(df_reviews.columns)}')
print(f'Missing:\n{df_reviews.isnull().sum()}')

# Drop unnamed index column if present
if 'Unnamed: 0' in df_reviews.columns:
    df_reviews = df_reviews.drop('Unnamed: 0', axis=1)

# Rename columns for consistency
df_reviews.columns = [c.strip().replace(' ', '_') for c in df_reviews.columns]
print(f'\nRenamed columns: {list(df_reviews.columns)}')

In [ ]:
# ============================================================
# 2.2.2 Handle missing values in reviews
# ============================================================
# Find the review text column
text_cols = [c for c in df_reviews.columns if 'review' in c.lower() and 'text' in c.lower()]
if not text_cols:
    text_cols = [c for c in df_reviews.columns if 'review' in c.lower()]
review_text_col = text_cols[0] if text_cols else None

print(f'Review text column: {review_text_col}')

# Drop rows with no review text
if review_text_col:
    before = len(df_reviews)
    df_reviews = df_reviews.dropna(subset=[review_text_col]).copy()
    print(f'Dropped {before - len(df_reviews)} rows with no review text')

# Fill other missing values
for col in df_reviews.select_dtypes(include='object').columns:
    df_reviews[col] = df_reviews[col].fillna('Unknown')
for col in df_reviews.select_dtypes(include='number').columns:
    df_reviews[col] = df_reviews[col].fillna(df_reviews[col].median())

print(f'After cleaning: {df_reviews.shape}')
print(f'Missing: {df_reviews.isnull().sum().sum()}')

In [ ]:
# ============================================================
# 2.2.3 Clean review text
# ============================================================
if review_text_col:
    df_reviews[review_text_col] = (
        df_reviews[review_text_col]
        .str.lower()
        .str.strip()
    )
    df_reviews['ReviewLength'] = df_reviews[review_text_col].str.split().str.len()
    print(f'Avg review length: {df_reviews["ReviewLength"].mean():.1f} words')
    print(f'Min: {df_reviews["ReviewLength"].min()}, Max: {df_reviews["ReviewLength"].max()}')

# Save cleaned reviews
df_reviews.to_csv(DATA_PROCESSED / 'reviews_clean.csv', index=False)
print('\n✅ Saved: data/processed/reviews_clean.csv')

### 2.3 Clean Amazon Products

In [ ]:
# ============================================================
# 2.3.1 Clean Amazon Products
# ============================================================
print(f'Before: {df_amazon.shape}')
print(f'Columns: {list(df_amazon.columns)}')

# Drop unnamed index
if 'Unnamed: 0' in df_amazon.columns:
    df_amazon = df_amazon.drop('Unnamed: 0', axis=1)

# Clean price columns (remove ₹ and commas)
for price_col in ['discount_price', 'actual_price']:
    if price_col in df_amazon.columns:
        df_amazon[price_col] = (
            df_amazon[price_col]
            .astype(str)
            .str.replace('₹', '', regex=False)
            .str.replace(',', '', regex=False)
            .str.strip()
        )
        df_amazon[price_col] = pd.to_numeric(df_amazon[price_col], errors='coerce')

# Clean ratings (extract numeric part)
if 'ratings' in df_amazon.columns:
    df_amazon['ratings'] = (
        df_amazon['ratings']
        .astype(str)
        .str.extract(r'(\d+\.?\d*)')[0]
    )
    df_amazon['ratings'] = pd.to_numeric(df_amazon['ratings'], errors='coerce')

# Clean no_of_ratings
if 'no_of_ratings' in df_amazon.columns:
    df_amazon['no_of_ratings'] = (
        df_amazon['no_of_ratings']
        .astype(str)
        .str.replace(',', '', regex=False)
        .str.strip()
    )
    df_amazon['no_of_ratings'] = pd.to_numeric(df_amazon['no_of_ratings'], errors='coerce')

# Drop rows with no name or category
df_amazon = df_amazon.dropna(subset=['name']).copy()

# Fill missing
df_amazon['ratings'] = df_amazon['ratings'].fillna(df_amazon['ratings'].median())
df_amazon['no_of_ratings'] = df_amazon['no_of_ratings'].fillna(0).astype(int)
df_amazon['discount_price'] = df_amazon['discount_price'].fillna(df_amazon['actual_price'])
df_amazon['actual_price'] = df_amazon['actual_price'].fillna(df_amazon['discount_price'])

print(f'After cleaning: {df_amazon.shape}')
print(f'Missing: {df_amazon.isnull().sum().sum()}')
df_amazon.head(3)

In [ ]:
# ============================================================
# 2.3.2 Extract categories & compute price features
# ============================================================
# Find category column
cat_col = None
for c in df_amazon.columns:
    if 'categ' in c.lower() or 'main' in c.lower():
        cat_col = c
        break

if cat_col:
    print(f'Category column: {cat_col}')
    print(f'Unique categories: {df_amazon[cat_col].nunique()}')
    print(df_amazon[cat_col].value_counts().head(10))

# Price range bins
if 'actual_price' in df_amazon.columns:
    df_amazon['PriceRange'] = pd.cut(
        df_amazon['actual_price'],
        bins=[0, 500, 2000, 10000, float('inf')],
        labels=['Low', 'Medium', 'High', 'Premium']
    )
    print(f'\nPrice Range Distribution:')
    print(df_amazon['PriceRange'].value_counts())

# Discount percentage
if 'discount_price' in df_amazon.columns and 'actual_price' in df_amazon.columns:
    df_amazon['DiscountPct'] = (
        (df_amazon['actual_price'] - df_amazon['discount_price']) / df_amazon['actual_price'] * 100
    ).clip(0, 100).round(1)

# Save cleaned amazon
df_amazon.to_csv(DATA_PROCESSED / 'amazon_clean.csv', index=False)
print('\n✅ Saved: data/processed/amazon_clean.csv')

---
## PART 3: FEATURE ENGINEERING — RFM & Customer Features

In [ ]:
# ============================================================
# 3.1 Compute RFM (Recency, Frequency, Monetary)
# ============================================================
# Reference date = day after last transaction
reference_date = df_retail['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f'Reference date: {reference_date}')

rfm = df_retail.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('TotalPrice', 'sum')
).reset_index()

rfm['Customer ID'] = rfm['Customer ID'].astype(int)

print(f'\nRFM Table: {rfm.shape}')
print(rfm.describe().round(2))
rfm.head()

In [ ]:
# ============================================================
# 3.2 Churn Label (Recency > 90 days = churned)
# ============================================================
CHURN_THRESHOLD = 90  # days
rfm['ChurnLabel'] = (rfm['Recency'] > CHURN_THRESHOLD).astype(int)

churn_rate = rfm['ChurnLabel'].mean() * 100
print(f'Churn threshold: {CHURN_THRESHOLD} days')
print(f'Churned customers: {rfm["ChurnLabel"].sum()} ({churn_rate:.1f}%)')
print(f'Active customers: {(rfm["ChurnLabel"]==0).sum()} ({100-churn_rate:.1f}%)')

In [ ]:
# ============================================================
# 3.3 Additional Customer Features
# ============================================================
# Average order value
rfm['AvgOrderValue'] = (rfm['Monetary'] / rfm['Frequency']).round(2)

# Customer first/last purchase dates
date_features = df_retail.groupby('Customer ID').agg(
    FirstPurchase=('InvoiceDate', 'min'),
    LastPurchase=('InvoiceDate', 'max'),
    TotalItems=('Quantity', 'sum'),
    UniqueProducts=('StockCode', 'nunique'),
    UniqueCountries=('Country', 'nunique')
).reset_index()

date_features['Customer ID'] = date_features['Customer ID'].astype(int)
date_features['DaysSinceFirstPurchase'] = (reference_date - date_features['FirstPurchase']).dt.days
date_features['CustomerLifespan'] = (date_features['LastPurchase'] - date_features['FirstPurchase']).dt.days

# Favorite category (most purchased product description)
fav_product = (
    df_retail.groupby(['Customer ID', 'Description'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .drop_duplicates(subset='Customer ID', keep='first')
    [['Customer ID', 'Description']]
    .rename(columns={'Description': 'FavoriteProduct'})
)
fav_product['Customer ID'] = fav_product['Customer ID'].astype(int)

# Country
customer_country = (
    df_retail.groupby('Customer ID')['Country']
    .agg(lambda x: x.mode()[0])
    .reset_index()
)
customer_country['Customer ID'] = customer_country['Customer ID'].astype(int)

# Merge all
customer_features = (
    rfm
    .merge(date_features[['Customer ID', 'TotalItems', 'UniqueProducts', 
                          'DaysSinceFirstPurchase', 'CustomerLifespan']], on='Customer ID')
    .merge(fav_product, on='Customer ID', how='left')
    .merge(customer_country, on='Customer ID', how='left')
)

print(f'Customer features table: {customer_features.shape}')
print(f'Columns: {list(customer_features.columns)}')
customer_features.head()

In [ ]:
# ============================================================
# 3.4 RFM Visualizations
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

rfm['Recency'].hist(bins=50, ax=axes[0], color='#3498db', edgecolor='white')
axes[0].set_title('Recency Distribution')
axes[0].set_xlabel('Days since last purchase')
axes[0].axvline(x=CHURN_THRESHOLD, color='red', linestyle='--', label=f'Churn = {CHURN_THRESHOLD}d')
axes[0].legend()

rfm['Frequency'].clip(0, 50).hist(bins=50, ax=axes[1], color='#2ecc71', edgecolor='white')
axes[1].set_title('Frequency Distribution')
axes[1].set_xlabel('Number of orders')

rfm['Monetary'].clip(0, 5000).hist(bins=50, ax=axes[2], color='#e67e22', edgecolor='white')
axes[2].set_title('Monetary Distribution')
axes[2].set_xlabel('Total spend (£)')

plt.suptitle('RFM Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/figures/etl_rfm_distributions.png', dpi=150)
plt.show()

---
## PART 4: LOAD — Populate MySQL & Generate Dataset

In [ ]:
# ============================================================
# 4.1 Connect to MySQL
# ============================================================
from sqlalchemy import create_engine, text

# UPDATE THESE with your phpMyAdmin credentials
DB_USER = 'root'
DB_PASS = ''  # default XAMPP has no password
DB_HOST = 'localhost'
DB_PORT = 3306
DB_NAME = 'ecommerce_dm'

# Create database if not exists
engine_base = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/')
with engine_base.connect() as conn:
    conn.execute(text(f'CREATE DATABASE IF NOT EXISTS {DB_NAME}'))
    conn.commit()
print(f'Database {DB_NAME} ready ✅')

# Connect to the database
engine = create_engine(f'mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}')
print(f'Connected to MySQL ✅')

In [ ]:
# ============================================================
# 4.2 Create Schema (run SQL script)
# ============================================================
schema_path = Path('../sql/create_schema.sql')
schema_sql = schema_path.read_text(encoding='utf-8')

# Split and execute each statement
with engine.connect() as conn:
    for statement in schema_sql.split(';'):
        stmt = statement.strip()
        if stmt and not stmt.startswith('--') and not stmt.startswith('USE') and not stmt.startswith('CREATE DATABASE'):
            try:
                conn.execute(text(stmt))
            except Exception as e:
                if 'already exists' not in str(e).lower() and 'duplicate' not in str(e).lower():
                    print(f'Warning: {e}')
    conn.commit()
print('Schema created ✅')

In [ ]:
# ============================================================
# 4.3 Populate Dim_Time
# ============================================================
dates = df_retail['InvoiceDate'].dt.date.unique()
dim_time = pd.DataFrame({'FullDate': pd.to_datetime(sorted(dates))})
dim_time['Day'] = dim_time['FullDate'].dt.day
dim_time['DayOfWeek'] = dim_time['FullDate'].dt.day_name()
dim_time['Month'] = dim_time['FullDate'].dt.month
dim_time['MonthName'] = dim_time['FullDate'].dt.month_name()
dim_time['Quarter'] = dim_time['FullDate'].dt.quarter
dim_time['Year'] = dim_time['FullDate'].dt.year
dim_time['IsWeekend'] = dim_time['FullDate'].dt.dayofweek.isin([5, 6]).astype(int)
dim_time['IsHoliday'] = 0  # Can be enriched later
dim_time.index = range(1, len(dim_time) + 1)
dim_time.index.name = 'TimeID'

dim_time.to_sql('Dim_Time', engine, if_exists='replace', index=True)
print(f'Dim_Time: {len(dim_time)} rows loaded ✅')

In [ ]:
# ============================================================
# 4.4 Populate Dim_Location
# ============================================================
countries = df_retail['Country'].unique()

# Map countries to regions
region_map = {
    'United Kingdom': 'Europe', 'France': 'Europe', 'Germany': 'Europe',
    'Spain': 'Europe', 'Italy': 'Europe', 'Netherlands': 'Europe',
    'Belgium': 'Europe', 'Switzerland': 'Europe', 'Portugal': 'Europe',
    'Norway': 'Europe', 'Sweden': 'Europe', 'Denmark': 'Europe',
    'Finland': 'Europe', 'Austria': 'Europe', 'Ireland': 'Europe',
    'Poland': 'Europe', 'Czech Republic': 'Europe', 'Greece': 'Europe',
    'Iceland': 'Europe', 'Malta': 'Europe', 'Cyprus': 'Europe',
    'Lithuania': 'Europe', 'Channel Islands': 'Europe', 'EIRE': 'Europe',
    'European Community': 'Europe',
    'USA': 'North America', 'Canada': 'North America',
    'Brazil': 'South America',
    'Australia': 'Oceania',
    'Japan': 'Asia', 'Singapore': 'Asia', 'Hong Kong': 'Asia',
    'Israel': 'Middle East', 'Lebanon': 'Middle East',
    'United Arab Emirates': 'Middle East', 'Bahrain': 'Middle East',
    'Saudi Arabia': 'Middle East',
    'South Africa': 'Africa', 'Nigeria': 'Africa',
    'RSA': 'Africa'
}

dim_location = pd.DataFrame({
    'Country': countries,
    'Region': [region_map.get(c, 'Other') for c in countries],
    'City': 'N/A'  # Not available in this dataset
})
dim_location.index = range(1, len(dim_location) + 1)
dim_location.index.name = 'LocationID'

dim_location.to_sql('Dim_Location', engine, if_exists='replace', index=True)
print(f'Dim_Location: {len(dim_location)} rows loaded ✅')

In [ ]:
# ============================================================
# 4.5 Populate Dim_Product
# ============================================================
products = df_retail.groupby('StockCode').agg(
    ProductName=('Description', 'first'),
    AvgPrice=('Price', 'mean')
).reset_index()

# Assign price ranges
products['PriceRange'] = pd.cut(
    products['AvgPrice'],
    bins=[0, 2, 5, 15, float('inf')],
    labels=['Low', 'Medium', 'High', 'Premium']
)

products['Category'] = 'General'  # Will be enriched with CNN later
products['SubCategory'] = 'N/A'
products['AvgRating'] = 0.0  # Will be filled from reviews
products['ImageCategory'] = None  # Will be filled from CNN

products.index = range(1, len(products) + 1)
products.index.name = 'ProductID'

# Create mapping for StockCode → ProductID
product_map = dict(zip(products['StockCode'], products.index))

products_to_sql = products.drop('StockCode', axis=1)
products['ProductID'] = products.index
products_to_sql.to_sql('Dim_Product', engine, if_exists='replace', index=True)
print(f'Dim_Product: {len(products)} rows loaded ✅')

In [ ]:
# ============================================================
# 4.6 Populate Dim_Customer
# ============================================================
dim_customer = customer_features[[
    'Customer ID', 'Country', 'Recency', 'Frequency', 'Monetary',
    'AvgOrderValue', 'ChurnLabel'
]].copy()
dim_customer = dim_customer.rename(columns={'Customer ID': 'CustomerID'})
dim_customer['AgeGroup'] = 'Unknown'  # Not available in retail dataset
dim_customer['JoinDate'] = None  # Will compute from first purchase
dim_customer['RFM_Segment'] = None  # Will be filled by K-Means in notebook 03

dim_customer.to_sql('Dim_Customer', engine, if_exists='replace', index=False)
print(f'Dim_Customer: {len(dim_customer)} rows loaded ✅')

In [ ]:
# ============================================================
# 4.7 Populate Fact_Orders
# ============================================================
# Create time mapping
time_map = dict(zip(
    dim_time['FullDate'].dt.date,
    dim_time.index if isinstance(dim_time.index, pd.RangeIndex) else range(1, len(dim_time)+1)
))

# Create location mapping
loc_map = dict(zip(dim_location['Country'], dim_location.index))

fact_orders = df_retail[['Invoice', 'Customer ID', 'StockCode', 'InvoiceDate',
                          'Country', 'Quantity', 'Price', 'TotalPrice']].copy()
fact_orders = fact_orders.rename(columns={
    'Invoice': 'InvoiceNo',
    'Customer ID': 'CustomerID',
    'Price': 'UnitPrice'
})
fact_orders['CustomerID'] = fact_orders['CustomerID'].astype(int)
fact_orders['ProductID'] = fact_orders['StockCode'].map(product_map)
fact_orders['TimeID'] = fact_orders['InvoiceDate'].dt.date.map(time_map)
fact_orders['LocationID'] = fact_orders['Country'].map(loc_map)
fact_orders['Discount'] = 0

# Drop unmapped rows and helper columns
fact_orders = fact_orders.dropna(subset=['ProductID', 'TimeID', 'LocationID'])
fact_orders['ProductID'] = fact_orders['ProductID'].astype(int)
fact_orders['TimeID'] = fact_orders['TimeID'].astype(int)
fact_orders['LocationID'] = fact_orders['LocationID'].astype(int)

fact_orders_sql = fact_orders[['InvoiceNo', 'CustomerID', 'ProductID', 'TimeID',
                                'LocationID', 'Quantity', 'UnitPrice', 'TotalPrice', 'Discount']]

fact_orders_sql.to_sql('Fact_Orders', engine, if_exists='replace', index=False)
print(f'Fact_Orders: {len(fact_orders_sql)} rows loaded ✅')

In [ ]:
# ============================================================
# 4.8 Generate Merged Customer Dataset (COURSE REQUIREMENT)
# ============================================================
print('Generating merged dataset from Data Warehouse...')

generated_dataset = customer_features.copy()
generated_dataset = generated_dataset.rename(columns={'Customer ID': 'CustomerID'})

# Add computed features
generated_dataset['TotalOrders'] = generated_dataset['Frequency']
generated_dataset['TotalSpent'] = generated_dataset['Monetary']
generated_dataset['AvgItemsPerOrder'] = (
    generated_dataset['TotalItems'] / generated_dataset['Frequency']
).round(2)

print(f'Generated dataset shape: {generated_dataset.shape}')
print(f'Columns: {list(generated_dataset.columns)}')
print(f'\nChurn distribution:\n{generated_dataset["ChurnLabel"].value_counts()}')

# Save
generated_dataset.to_csv(DATA_GENERATED / 'customer_features.csv', index=False)
print(f'\n✅ SAVED: data/generated/customer_features.csv')
print('   ↑ This is the NEW GENERATED DATASET created through the Data Warehouse (course requirement)')

generated_dataset.head()

In [ ]:
# ============================================================
# 4.9 Verify Data in MySQL
# ============================================================
print('=== DATA WAREHOUSE SUMMARY ===')
tables = ['Dim_Customer', 'Dim_Product', 'Dim_Time', 'Dim_Location', 'Fact_Orders']
for table in tables:
    count = pd.read_sql(f'SELECT COUNT(*) as cnt FROM {table}', engine).iloc[0, 0]
    print(f'  {table}: {count:,} rows')

print('\n✅ ETL Pipeline Complete!')

---
## ETL Summary

| Step | Input | Output |
|------|-------|--------|
| Extract | 3 raw datasets | Loaded DataFrames |
| Transform | Raw data | Cleaned + feature-engineered data |
| Load | Clean data → MySQL Star Schema | 4 Dimensions + 1 Fact table |
| Generate | DW queries → merged dataset | `customer_features.csv` |

### Files Created:
- `data/processed/retail_clean.csv`
- `data/processed/reviews_clean.csv`
- `data/processed/amazon_clean.csv`
- `data/processed/retail_cancellations.csv`
- `data/generated/customer_features.csv` ← **Generated dataset (course requirement)**